# NastaliqOps: LoRA fine tuning on Colab

Runs the train, evaluate and register stages of the DVC pipeline.
Everything before this (generate, ingest, merge, validate, preprocess,
split) already ran on the laptop and the data is on the DVC remote.

Run the cells in order. Nothing is stored in this notebook, the tokens
are typed in at run time and live only in the session.


## 1. Check the runtime


In [ ]:
import torch

print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Runtime > Change runtime type > T4 GPU, then rerun.")


## 2. Clone the repo


In [ ]:
!git clone https://github.com/MehrmahKhan/nastaliq-ocr.git
%cd nastaliq-ocr


## 3. Install

Takes a few minutes.


In [ ]:
# requirements.txt is the pinned set for docker on python 3.11. colab
# runs a newer python where several of those exact pins have no wheels,
# and one unresolvable line makes pip abort the whole file.
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

# colab ships torchao 0.10.0. peft's lora dispatcher probes torchao and
# raises ImportError when it finds a version below 0.16, which kills
# get_peft_model. nothing here uses torchao, so removing it is the fix.
!pip uninstall -q -y torchao


## 4. Credentials

Paste the DagsHub token when prompted. It is not echoed and not saved.


In [ ]:
import os
from getpass import getpass

DAGSHUB_USER = "MehrmahKhan"
DAGSHUB_REPO = "nastaliq-ocr"
token = getpass("DagsHub token: ")

os.environ["DAGSHUB_USER"] = DAGSHUB_USER
os.environ["DAGSHUB_REPO"] = DAGSHUB_REPO
os.environ["DAGSHUB_TOKEN"] = token
os.environ["MLFLOW_TRACKING_URI"] = (
    "https://dagshub.com/" + DAGSHUB_USER + "/" + DAGSHUB_REPO + ".mlflow"
)
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = token
os.environ["MLFLOW_EXPERIMENT"] = "nastaliq-ops"

print("environment set")


## 5. Point DVC at the remote and pull the data


In [ ]:
endpoint = "https://dagshub.com/" + DAGSHUB_USER + "/" + DAGSHUB_REPO + ".s3"

!dvc remote modify --local origin endpointurl {endpoint}
!dvc remote modify --local origin access_key_id {token}
!dvc remote modify --local origin secret_access_key {token}
!dvc pull data/splits data/processed


In [ ]:
import json

handle = open("reports/splits.json", "r", encoding="utf-8")
print(json.load(handle))
handle.close()


## 6. Train

3 epochs over roughly 1270 lines. Twenty to forty minutes on a T4.
The run and its parameters are logged to MLflow on DagsHub as it goes.


In [ ]:
!python -m nastaliqops.models.train_lora


## 7. Evaluate on the frozen golden set


In [ ]:
!python -m nastaliqops.models.evaluate


In [ ]:
handle = open("reports/metrics.json", "r", encoding="utf-8")
metrics = json.load(handle)
handle.close()

for name in metrics:
    print(name, ":", metrics[name])


## 8. Register the adapter and run the promotion gate

Exit code 0 means the challenger was promoted to Production.
Exit code 2 means it was registered but rejected, and the champion stays.
Both outcomes are correct behaviour, a rejection is the gate working.


In [ ]:
!python -m nastaliqops.models.register; echo "exit code: $?"


## 9. Push the adapter back to the remote


In [ ]:
!dvc add artifacts/adapter
!dvc push


## 10. Download the pointer files

Commit these on the laptop so the repo points at the trained adapter.


In [ ]:
from google.colab import files

to_download = ["artifacts/adapter.dvc", "reports/metrics.json"]
index = 0
while index < len(to_download):
    path = to_download[index]
    if os.path.exists(path):
        files.download(path)
    else:
        print("missing:", path)
    index = index + 1
